<a href="https://colab.research.google.com/github/loisvanessaadams-create/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY").strip()

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
           temperature=0.7, max_tokens=500):
  response = client.chat.completions.create(
      model=MODEL
      ,messages=[
        {"role": "system", "content": system_prompt},
         {"role": "user",   "content": user_prompt},
        ],
      temperature=temperature,
      max_tokens=max_tokens,
      )
  print("Token usage:", response.usage)
  return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?
answer = ask_llm("What is artificial intelligence")
print(answer)



Token usage: CompletionUsage(completion_tokens=500, prompt_tokens=45, total_tokens=545, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057401318, prompt_time=0.009554412, completion_time=1.744495737, total_time=1.754050149)
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data, experiences, and interactions, enabling them to improve their performance over time.
2. **Problem-solving**: AI systems can analyze complex problems, identify patterns, and provide solutions or recommendations.
3. **Reasoning**: AI systems can draw inferences, make decisions, and take actions based on available data and knowledge.
4. **Perception**: AI systems can interpret and understand data from sensors, such as images, speech, and text.
5. **Natural Language Processing (NLP)**: AI systems can understand, generate, and process human language

[link text](https://)**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. The system role gives the model overall instructions about how it should behave or respond, while the user role contains the specific request from the user. An example for a system prompt could be  "You an intro to AI assistant. Give clear and easy to understand answers . For a user prompt it could be explain how concepts in computer vision can be brought to the real world.

2. A token is a small piece of text that the model processes, it can be a word, a part of a word or even a punctuation. API providers bill per token because different requests can need different levels of computation. A short question and answer session would use fewer tokens than a long document and response, even though they are both technically one API request.

### Part 1.2 — Temperature: the randomness dial

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
question = "Suggest a name for a savings product for market traders in Accra."
print("TEMPERATURE = 0.0")
for i in range(5):
  answer = ask_llm(
      question,
      temperature = 0.0,
      max_tokens=100)
  print(f"{i+1}.{answer}")
  print()

print("Temperature = 1.2")
for i in range(5):
  answer = ask_llm(
      question,
      temperature = 1.2,
      max_tokens = 100
  )
  print(f"{i + 1}.{answer}")
  print()

# TODO: Print all 10 answers, grouped by temperature.

TEMPERATURE = 0.0
Token usage: CompletionUsage(completion_tokens=100, prompt_tokens=56, total_tokens=156, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057009665, prompt_time=0.003462909, completion_time=0.299464592, total_time=0.302927501)
1.Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kurom**: "Sika" means "money" in the Akan language, and "Kurom" means "save" or "keep

Token usage: CompletionUsage(completion_tokens=100, prompt_tokens=56, total_tokens=156, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057407764, prompt_time=0.017404415, completion_time=0.326128133, total_time=0.343532548)
2.Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature = 0.0, the responses were much more consisted and often had similar names such as "Makola Save" and "Trader's Treasure." At temperature = 1.2, the responses were more varied and creative while more names and wording varying. For the loan decision system, a low twmpreature 0.0 is more appropriate because teh system should be consistent, factual and less likely to introduce unnecessary variation or unsupported details.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
SUMMARY_PROMPT_V1 ="Summarize this: "

#   Run it on L002 and L006. Read the output critically
#for V1: L002
v1_l002 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}",temperature = 0.0)

#for V1: L006
v1_l006 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}",temperature = 0.0)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
SUMMARY_SYSTEM_PROMPT_V2 = """ You are an assistant to a micro finance loan officer. Summarize loan applications clearly and accurately.
Be factual and neutral.USE information stored in the application.Do not invent or assume missing details.
Keep the summary to 3-4 sentences.
"""
SUMMARY_PROMPT_V2 = "Summarize this loan application:"
#for V2: L002
v2_l002 = ask_llm(f"{SUMMARY_PROMPT_V2}\n\n{LETTERS['L002']}",system_prompt = SUMMARY_SYSTEM_PROMPT_V2,temperature = 0.0)
print(v2_l002)
v2_l006 = ask_llm(f"{SUMMARY_PROMPT_V2}\n\n{LETTERS['L006']}", system_prompt = SUMMARY_SYSTEM_PROMPT_V2, temperature = 0.0)

#for L002
print("V1:")
print(v1_l002)
print("\nV2:")
print(v2_l002)

#for L006
print("V1:")
print(v1_l006)
print("\nV2:")
print(v2_l006)

#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

Token usage: CompletionUsage(completion_tokens=68, prompt_tokens=134, total_tokens=202, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.056899867, prompt_time=0.006726742, completion_time=0.250759769, total_time=0.257486511)
Token usage: CompletionUsage(completion_tokens=94, prompt_tokens=136, total_tokens=230, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057708493, prompt_time=0.008480021, completion_time=0.314830594, total_time=0.323310615)
Token usage: CompletionUsage(completion_tokens=78, prompt_tokens=178, total_tokens=256, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.058471075, prompt_time=0.022365757, completion_time=0.223976769, total_time=0.246342526)
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. The loan is intended to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season, which will enable him to repa

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*

*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V1 sometimes interpreted or added information that was not directly stated in the applications. For example, for L006, V1 said Kofi had "no prior experience" even though the letter only said that he had not started the proposed businesses. It also describes his "personal trustworthiness as collateral," although the application explicity stated that he had no collateral. V2 improved this by stating that he "has not yet initiatied any of these ventures" and "does not have collateral to offer." V2 was therefore more factual and closer to the information provided in the original letters.

2. The instruction "no innvented details is essential because adding information that is not in a loan application could misrepresent an applicant and potentially influence the loan officer's assessment. This type of LLM failure is called hallucination where the model generated information that is unsupported or not in the probided input.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [6]:
import json
import pandas as pd

EXTRACT_PROMPT = '''
You are extracting structured information from a loan application letter.
REturn only a valid JSOn object with exactly these keys:
{
  "applicant_name": string,
  "amount_ghs": number,
  "purpose": string,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}
Rules:
- Use only information explicitly ststed in the letter.
-If a field is not stated, use null.
- Do not guess.
- Do not include explanations.
- Do not include markdown or ``json fences.
- has_collateral_or_guarantor should be true if the applicant mentions either collateral or a guarantor,otherwise false.

Examples:
Letter:
My name is Lois Adams. I am requesting GHS 12000 to expand my bookstore business.
The bookstore currently makes about 4000 profit per month.
I can povide my car as collateral and plan to repay the loan within 12 months.

Output:
{
"applicant_name": "Ama Mensah",
"amount_ghs": 12000,
"purpose": "Expand bookstore business",
"monthly_profit_ghs": 4000,
"has_collateral_or_guarantor": true,
"repayment_months": 12
}
Now extract the information from this letter:
'''

# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
  response = ask_llm(
      f"{EXTRACT_PROMPT}\n\n{letter_text}",
      temperature = 0.0
  )

#Removing possible markdown fences
  cleaned = response.strip()
  cleaned = cleaned.replace("```json","")
  cleaned = cleaned.replace("```","")
  cleaned = cleaned.strip()
  try:
    result = json.loads(cleaned)
    return result

  except json.JSONDecodeError:
    print("Warning: Could not parser JSON response.")
    print("Raw response:")
    print(response)
    return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = []
for letter_id, letter_text in LETTERS.items():
  extracted = extract_fields(letter_text)

  if extracted is not None:
    extracted["letter_id"] = letter_id
    results.append(extracted)

df_extracted = pd.DataFrame(results)
df_extracted = df_extracted[[
    "letter_id",
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]
        ]
display(df_extracted)

Token usage: CompletionUsage(completion_tokens=67, prompt_tokens=445, total_tokens=512, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.056996093, prompt_time=0.171256404, completion_time=0.132083962, total_time=0.303340366)
Token usage: CompletionUsage(completion_tokens=64, prompt_tokens=405, total_tokens=469, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.056881034, prompt_time=0.033315545, completion_time=0.126881723, total_time=0.160197268)
Token usage: CompletionUsage(completion_tokens=71, prompt_tokens=459, total_tokens=530, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057329187, prompt_time=0.046557172, completion_time=0.161971078, total_time=0.20852825)
Token usage: CompletionUsage(completion_tokens=72, prompt_tokens=425, total_tokens=497, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057585852, prompt_time=0.045768037, completion_time=0.165670732, total_time=0.211438769)
T

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,poultry farm at Nsawam for feed and 500 new la...,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
The few-shot example should not come from the six letters being processed because it would expose the model to part of the actual evaluation data. Using a separate example makes the extraction test fairer and shows the model can apply the pattern to new letters rather than copy information it has already seen.

*2. Why "use null, do not guess" — what did the model do without that instruction?*

*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [Double-click to edit]

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [7]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT = """
You are an assistant supporting a human microfinance loan officer.
Using only the original loan application and the extracted JSON provided below, prepare a decision-support brief.
Your brief must contain exactly these four sections:
1.Strengths
- Use bullet point.
- Include only strengths supported by the application.
r
2. Risks/Red Flags
- Use bullet points.
- Identify concerns supported by the application.
- Do not invent risks or facts.

3.Missing Information
- Use bullet points.
-Identify important informtion the loan officer should request or verify.

4.Suggested Next Step
- Suggest an appropriate section such as requesting documents, inviting the applicant for an interview, or flagging the application for senior review.
- Do not approve or reject the application.

Be factual,neutral and concise
Do not invent missing financial information, collatoral, income or repayment details. Final lending decisons must be made by a human loan officer.
"""
# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
extracted_data = ((df_extracted).set_index("letter_id").to_dict(orient = "index"))

briefs = {}
for letter_id, letter_text in LETTERS.items():
  user_prompt = f"""
Original loan application:
{letter_text}
Extracted JSON:
{json.dumps(extracted_data[letter_id],indent = 2)}
Prepare the decision support brief.
"""
  briefs[letter_id] = ask_llm(
          user_prompt,
          system_prompt = BRIEF_PROMPT,
          temperature = 0.0,
          max_tokens = 500
      )

for letter_id in ["L001","L002","L006"]:
    print(f"\n {letter_id}")
    print(briefs[letter_id])



Token usage: CompletionUsage(completion_tokens=288, prompt_tokens=445, total_tokens=733, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172813464, prompt_time=0.021493155, completion_time=0.826252417, total_time=0.847745572)
Token usage: CompletionUsage(completion_tokens=296, prompt_tokens=398, total_tokens=694, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.170884707, prompt_time=0.019301671, completion_time=0.901261901, total_time=0.920563572)
Token usage: CompletionUsage(completion_tokens=355, prompt_tokens=463, total_tokens=818, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.223371616, prompt_time=0.023854469, completion_time=2.126005531, total_time=2.14986)
Token usage: CompletionUsage(completion_tokens=337, prompt_tokens=430, total_tokens=767, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.171985366, prompt_time=0.024650084, completion_time=1.166755943, total_time=1.191406027)


**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. Yes, the system generally identified the important differences between the applications. For  L003, the strong points include an established registered business, GHS 2800 average monthly profit, a GHS 5000 fixed deposit that can be pledged, sales records, and a clear repayment plan. For L006, it correctly identified major risks such as having no existing business, no collateral or guarantor, no stated income, and relying on three new businesses becoming successfull within one year. One weakness is that the L006 brief treated being "young and full of energy" as a strength, even though this does not provide evidence of repayment ability.

2. Practically, the model should not output "approve" or "reject" because it may misunderstand, omit, or invent information, so a human officer must verify the evidence before making the final decision. Ethically, an automated approval or rejection could unfairly affect applicants, especially if the model contains bias or interprets differences in language and writng as financial risk.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 6c7bd6d


In [8]:
%%writefile prompts.py

SUMMARY_SYSTEM_PROMPT = """
You are an assistant to a microfinance loan officer.
Summarize loan applications clearly and accurately.
Be factual and neutral.
USE only information stated in the application.
Do not invent or assume missing details.
Keep the summary to 3-4 sentences.
"""
SUMMARY_PROMPT = "Summarize this loan application:"

EXTRACT_PROMPT = """
You are extracting structured information from a loan application letter.
Return only a valid JSON object with exactly these keys:
{
  "applicant_name": string,
  "amount_ghs": number,
  "purpose": string,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}
Rules:
- Use only information explicitly stated in the letter.
- If a field is not stated, use null.
- Do not guess.
- Do not include explanations.
- Do not include markdown or JSON fences.
- has_collateral_or_guarantor should be true if the applicant mentions either collateral or a guarantor; otherwise false.

Example:
Letter:
My name is Lois Adams. I am requesting GHS 12000 to expand my bookstore business.
The bookstore currently makes about 4000 profit per month.
I can provide my car as collateral and plan to repay the loan within 12 months.

Output:
{
"applicant_name": "Lois Adams",
"amount_ghs": 12000,
"purpose": "Expand bookstore business",
"monthly_profit_ghs": 4000,
"has_collateral_or_guarantor": true,
"repayment_months": 12
}
Now extract the information from this letter:
"""

BRIEF_PROMPT = """
You are an assistant supporting a human microfinance loan officer.
Using only the original loan application and the extracted JSON provided, prepare a decision-support brief.
Your brief must contain exactly these four sections:

1.Strengths
- Use bullet points.
- Include only strengths supported by the application.

2. Risks/Red Flags
- Use bullet points.
- Identify concerns supported by the application.
- Do not invent risks or facts.

3.Missing Information
- Use bullet points.
-Identify important information the loan officer should request or verify.

4.Suggested Next Step
- Suggest an appropriate section such as requesting documents, inviting the applicant for an interview, or flagging the application for senior review.
- Do not approve or reject the application.

Be factual, neutral, and concise.
Do not invent missing financial information, collateral, income or repayment details.
Final lending decisions must be made by a human loan officer.
"""

Writing prompts.py


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [10]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

gold_letters = ["L001", "L003", "L006"]

extracted_lookup = (
    df_extracted.set_index("letter_id").to_dict(orient = "index")
)
comparison = []

for field in fields:
  row = {"field": field}
  correct = 0

  for letter_id in gold_letters:
    extracted_value = extracted_lookup[letter_id][field]
    gold_value = GOLD[letter_id][field]

    #Applicant names can be case-insensitive
    if field == "applicant name":
      match = (
          str(extracted_value).lower().strip()
          ==str(gold_value).lower().strip()
      )

      #checking null values
    elif pd.isna(extracted_value) and pd.isna(gold_value):
      match = True

    else:
      match = extracted_value == gold_value

    if match:
      row[letter_id] = "Correct"
      correct += 1
    else:
      row[letter_id] = "Incorrect"

  row["accuracy"] = correct / len(gold_letters)
  comparison.append(row)

accuracy_df = pd.DataFrame(comparison)
display(accuracy_df)

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

,field,L001,L003,L006,accuracy
0,applicant_name,Correct,Correct,Correct,1.0
1,amount_ghs,Correct,Correct,Correct,1.0
2,purpose,Incorrect,Incorrect,Incorrect,0.0
3,monthly_profit_ghs,Correct,Correct,Correct,1.0
4,has_collateral_or_guarantor,Correct,Correct,Correct,1.0
5,repayment_months,Correct,Correct,Correct,1.0


### Part 4.2 — Reliability: is the system consistent?

In [16]:
from enum import unique
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
def extract_fields(letter_text,temperature=0.0):
  response = ask_llm(
      f"{EXTRACT_PROMPT}\n\n{letter_text}",
      temperature = 0.0
  )
  cleaned = response.strip()
  cleaned = cleaned.replace("```json","")
  cleaned = cleaned.replace("```","")
  cleaned = cleaned.strip()

  try:
    result = json.loads(cleaned)
    return result
  except json.JSONDecodeError:
    print("Warning: Could not parse JSON response.")
    return None

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
results_temp_0 = []
results_temp_1 = []
#5 runs at temp=0.0
for i in range(5):
  result = extract_fields(
      LETTERS["L004"],
      temperature = 0.0
  )
  results_temp_0.append(result)

for i in range(5):
  result = extract_fields(
      LETTERS["L004"],
      temperature = 1.0
  )
  results_temp_1.append(result)
def check_reliability(results):
  valid_results = [
      result for tesult in results
      if result is not None
    ]
  valid_count = len(valid_results)

  unique_results = set(
        json.dumps(result, sort_keys = True)
        for result in valid_results
    )

  unique_count = len(unique_results)

  identical = (valid_count == 5 and unique_count == 1)

  return valid_count, unique_count, identical

valid_0,unique_0,identical_0 = check_reliability(results_temp_0)
valid_1,unique_1,identical_1 = check_reliability(results_temp_1)

print("Temperature = 0.0")
print(f"Valid results: ",valid_0)
print(f"Unique results: ",unique_0)
print(f"Identical results: ",identical_0)

print("Temperature = 1.0")
print(f"Valid results: ",valid_1)
print(f"Unique results: ",unique_1)
print(f"Identical results: ",identical_1)




#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

Token usage: CompletionUsage(completion_tokens=72, prompt_tokens=425, total_tokens=497, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057099405, prompt_time=0.051552776, completion_time=0.166925466, total_time=0.218478242)
Token usage: CompletionUsage(completion_tokens=72, prompt_tokens=425, total_tokens=497, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057350127, prompt_time=0.038890766, completion_time=0.161036743, total_time=0.199927509)
Token usage: CompletionUsage(completion_tokens=72, prompt_tokens=425, total_tokens=497, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172301296, prompt_time=0.020823081, completion_time=0.13486854, total_time=0.155691621)
Token usage: CompletionUsage(completion_tokens=72, prompt_tokens=425, total_tokens=497, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.057179469, prompt_time=0.023255215, completion_time=0.137232567, total_time=0.160487782)
T

### Part 4.3 — Hallucination probing

In [19]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
test1_prompt = f""" Here is the loan application:{LETTERS["L001"]}
What is the applicant's credit score?
Answer using only information in the application
"""
test1_output = ask_llm(
    test1_prompt,
    system_prompt = SUMMARY_SYSTEM_PROMPT_V2,
    temperature = 0.0

    )
print("Test 1 Output: ")
print(test1_output)
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
"TEST2"
irrelevant = """
The weather in Accra is cloudy,it might rain in the afternoon. Temperatures continue to be warm throughout the day."""

test2_output = extract_fields(
    irrelevant,
    temperature = 0.0
)
print("Test 2 Output: ")
print(test2_output)
print(json.dumps(test2_output,indent = 2))
# TODO: Record the outputs verbatim below and label each PASS or FAIL.
"""
Test 1
Output:"The applicants credit score was not mentioned in the application. The application only mentions that the applicant has saved GHS 2500 with the susu schemeover the past two years and has never missed a contribution, which siggests a good savings history but does not provide a credit score.
RESULT: PASS - The model correctly admitted that the credit score wasa absent instead of inventing one.
Test 2
Output: "{applicant_name": null, "amount_ghs": null, "purpose": null, "monthly_profit_ghs": null, "has_collateral_or_guarantor": false, "repayment_months": null}"
RESULT: PASS - The extractor returned nulls for information that was not present and did not fabricate an applicant.


Token usage: CompletionUsage(completion_tokens=60, prompt_tokens=233, total_tokens=293, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172079707, prompt_time=0.011181178, completion_time=0.134928228, total_time=0.146109406)
Test 1 Output: 
The applicant's credit score is not mentioned in the application. The application only mentions that the applicant has saved GHS 2,500 with the susu scheme over the past two years and has never missed a contribution, which suggests a good savings history, but does not provide a credit score.
Token usage: CompletionUsage(completion_tokens=49, prompt_tokens=343, total_tokens=392, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172918658, prompt_time=0.034444675, completion_time=0.077170688, total_time=0.111615363)
Test 2 Output: 
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': None}
{
  "applicant_name":

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*

*2. What did the reliability experiment show about temperature and production systems?*

*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
1. The extraction system achieved 83.3% overall exact-match accuracy, with 15 correct values out of 18 comparisons. Five fields had 100% accuracy, while purpose had 0% exact match accuracy. The purpose field was hardest because the model extracted the correct meaning but used different wording from the gold values. For example, L003 returned "purchase two industrial sewing machined and fabric stock ahead of the Christmad seasin," while the gold value was "industrial sewing machines and fabric stock."

2. The reliability experiment showed that both temperature = 0.0 and temperature = 1.0 produced 5/5 valid outputs with only one unique result.This suggests that the strict execution schema strongly constrained the even at the higher temperature. However, low temperature is still more appropriate for production because it reduces unnecessary randomness and supports consistent outputs.

3. The system did not hallucinate in the two adversarial tests. It correctly stated that the credit score was not provided, and it returned null values when given irrelevant text. Hallucination risk can be redued by explicitly telling the model not to guess, using null for missing information, validating structured outputs, usinf low temperature, and requiring human review before decisions are made.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*

*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*

*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
1. Fully automating loan decisions could unfairly harm applicants who have good businesses but communicate poorly in English or provide less polished written applications. The system might interpret weak writing as weak business ability, causing capable applicants to be treated as higher risk even when their actual financial situation is strong.

2. Sending loan letters to a foreign third-party API creates privacy and data-security risks because the letters contain personal and financial information. Before deployment, the institution should check how the provider stores, processes, retains and protects the data, whether the data may be used for model training, and whether cross-border processing compiles with Ghanaian data-protection requirements.

3. Two safeguards would be mandatory human review, so the AI only supports rather than makes the final lending decision, and an audit process where decisions and AI outputs are logged so applicants can challenge errors and officers can investigate possible bias.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?

2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?

3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?

4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**
1. Iterating on prompts is similar to tuning hyperparameters in Lab 3 because both involve testing different settings, evaluating the resulrs, and making changes to improve performance, The difference is that hyperparameters change how a model learns or trains, while prompt engineering changes the instructions given to an already-trained model without changing its internal weights.

2. I would not trust this system to run completely unattended. The result that influenced me most was the 83.3% exact match extraction accuracy, particularly the failure of the purpose field under exact matching. Although the extracted meanings were generally correct, this showsed that model outputs and evaluaton methods can still introduce errors, so human review remains necessary.

3. Based on my API calls, extracton used about 500 tokens, summarization about 250 tokens, and the decision-support brief about 700 tokens per application. This gives an estimate of roughly 1450 tokens per application, or about 1.45 million tokens for 1000 applications per month. At this scale,provider choice should consider token pricing, rate limits, reliability,privacy and whether the model provides enough accuracy for its cost.

4. For this task, using a foundation model through an API is more practical than training a model from scratch becasue the model understands language and can perform summarization, extraction, and reasoning with only carefully designed prompts. Training our model would require much more data, computing power, time and evaluation. However, a custom model could be more appropriate if the organization had a large amount of specialized training data, strict privacy requirements, needed full control over deployment, or required performance specifically optimized for its own lending process.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.